# GRPO on GSM8K — one-click smoke run

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hydaspex/grpo-math-reasoning/blob/main/notebooks/colab_grpo.ipynb)

Trains Qwen2.5-1.5B on GSM8K math reasoning with **GRPO** (Group Relative Policy Optimization) against verifiable rewards, alongside an **SFT control arm**, then compares both to the base model.

This notebook runs a deliberately **reduced smoke configuration**: it proves the pipeline works end to end on a free T4, but the step count is far too low to produce meaningful scores. Headline numbers come from a longer run on a bigger budget — see the closing section.

## 1. Runtime check

Use **Runtime → Change runtime type → T4 GPU** (or better). GRPO needs a CUDA GPU; there is no CPU fallback.

In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('bf16 supported:', torch.cuda.is_bf16_supported())

## 2. Optional Google Drive storage

Mount Drive to keep adapters and the MLflow store after the runtime is recycled.

In [ ]:
USE_GOOGLE_DRIVE = False
DRIVE_OUTPUT = '/content/drive/MyDrive/grpo-math-reasoning'

import os
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

## 3. Install Unsloth and a compatible vLLM

GRPO samples a group of completions at every step, so it leans on vLLM for fast generation. **Unsloth, vLLM, TRL and transformers are tightly version-coupled and the working combination shifts between releases** — pinning the wrong set is the single most common way this fails to start.

The pins below mirror Unsloth's own GRPO notebook at time of writing. If this cell errors, copy the install cell from [Unsloth's current GRPO notebook](https://github.com/unslothai/notebooks) rather than adjusting versions by hand.

In [ ]:
import torch

is_t4 = 'T4' in torch.cuda.get_device_name(0)
vllm_pin = 'vllm==0.11.2' if is_t4 else 'vllm==0.15.1'
print('Installing', vllm_pin)

!pip install -q --upgrade uv
!uv pip install -q --system unsloth unsloth_zoo {vllm_pin}
!uv pip install -q --system --no-deps transformers==4.56.2 trl==0.22.2

## 4. Clone the repository

Idempotent: removes any existing clone first, so the cell can be re-run safely.

In [ ]:
import shutil
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Hydaspex/grpo-math-reasoning.git'
BRANCH = 'main'
REPO_DIR = Path('/content/grpo-math-reasoning')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone -q --branch $BRANCH $REPO_URL $REPO_DIR
os.chdir(REPO_DIR)
!pip install -q --no-deps -e .

# A live kernel builds sys.path from .pth/editable hooks at interpreter
# startup, so an editable install run mid-session isn't always importable
# without this explicit fallback.
src_dir = str(REPO_DIR / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from mathrl.config import load_config
print('mathrl imported from', Path.cwd())

## 5. Configure the smoke run

Shrinks the dataset and step count so the whole notebook finishes in one session. `MAX_SAMPLES = None` and a few hundred GRPO steps is what a real run looks like.

In [ ]:
import yaml

MAX_SAMPLES = 200
GRPO_MAX_STEPS = 20

config_path = Path('configs/grpo_qwen25_1_5b.yaml')
config = yaml.safe_load(config_path.read_text())
config['data']['max_samples'] = MAX_SAMPLES
config['grpo']['max_steps'] = GRPO_MAX_STEPS
config['sft']['max_steps'] = 20

# Absolute sqlite URI: MLflow's default file store resolves './mlruns'
# relative to the working directory, which breaks after a runtime restart
# (cwd resets to /content). An absolute path keeps every stage writing to
# the same store.
MLFLOW_DB = (REPO_DIR / 'mlflow.db').resolve()
config['mlflow']['tracking_uri'] = f'sqlite:////{MLFLOW_DB}'

config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())

## 6. Prepare GSM8K data

Writes prompt-only records for GRPO, supervised records for the control arm, and a shared validation split — all from the same examples.

In [ ]:
!python scripts/prepare_data.py --config configs/grpo_qwen25_1_5b.yaml

## 7. SFT control arm

Trains on the same examples, targeting the same `<reasoning>`/`<answer>` format the GRPO reward pays out for. Without this arm, a GRPO improvement can't be distinguished from "any fine-tuning would have helped".

In [ ]:
!python scripts/train_sft.py --config configs/grpo_qwen25_1_5b.yaml

## 8. GRPO training

**Reward will sit near zero and barely move during this smoke run — that is expected, not a failure.** Correctness fires rarely for the first 100–200 steps; the format rewards carry the early signal. Killing a run that looks flat at step 20 is the most common mistake with GRPO.

Watch `reward` and `reward_std` in the logs rather than `loss`, which stays near zero by construction.

In [ ]:
!python scripts/train_grpo.py --config configs/grpo_qwen25_1_5b.yaml

## 9. Compare base, SFT and GRPO

Scores all three on the held-out split, with bootstrap confidence intervals and a one-sided paired McNemar test between adjacent arms.

In [ ]:
!python scripts/compare_models.py \
    --config configs/grpo_qwen25_1_5b.yaml \
    --sft-adapter outputs/qwen25-gsm8k-sft \
    --grpo-adapter outputs/qwen25-gsm8k-grpo \
    --batch-size 8

## 10. Review runs in MLflow

Every stage logged to the absolute sqlite store configured in step 5, so results survive a runtime restart.

In [ ]:
import mlflow

mlflow.set_tracking_uri(config['mlflow']['tracking_uri'])

experiments = mlflow.search_experiments()
print(f'Found {len(experiments)} experiments.')

runs = mlflow.search_runs(experiment_ids=[e.experiment_id for e in experiments])
wanted = ['tags.mlflow.runName', 'metrics.accuracy', 'metrics.format_valid']

if runs.empty:
    print(f'No runs found at {mlflow.get_tracking_uri()}.')
else:
    present = [c for c in wanted if c in runs.columns]
    if 'metrics.accuracy' in present:
        table = runs[present].rename(columns={'tags.mlflow.runName': 'run_name'})
        print(table.dropna(subset=['metrics.accuracy']).sort_values(
            'metrics.accuracy', ascending=False))
    else:
        print('Runs found but no accuracy metric yet. Columns:', runs.columns.tolist())

## Running the real experiment

This smoke run validates mechanics only. For numbers worth reporting:

- `max_samples: null` — the full ~7.5k GSM8K training split
- `grpo.max_steps: 250` or more — reward typically only starts climbing after 100–200 steps
- evaluate on the full validation split (`--limit 0`, the default)

That is a multi-hour job. A free Colab T4 fits the memory requirements for a 1.5B model but is slow; Kaggle offers roughly 30 GPU-hours per week free and is a better home for the full run. The scripts take `--config` and are platform-agnostic — only this notebook is Colab-specific.

Record the GPU, wall-clock, peak VRAM and config alongside the results.

**Watch for reward hacking**: reward climbing while KL divergence also climbs, with completions degenerating into repetitive text that games the format rewards without reasoning. If that appears, raise `grpo.beta` above `0.0` to pull the policy back toward the reference model.